## 대구

In [19]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# 1. 전처리 및 Anomaly가 적용된 데이터 불러오기
df = pd.read_csv('../data/polar_weather_preprocessed.csv', parse_dates=['Date'])

# 2. 대구 파생 변수 생성 (장보고기지 중심, Lag 14일 적용)
daegu_lag = 14
df['Jangbogo_Lag_D'] = df['Jangbogo_Temp_Mean_Anomaly'].shift(daegu_lag)
df['Jangbogo_Lag_D_MA3'] = df['Jangbogo_Temp_Mean_Anomaly'].shift(daegu_lag).rolling(window=3).mean()
df['Sejong_Lag_D'] = df['Sejong_Temp_Mean_Anomaly'].shift(daegu_lag)
df['Jangbogo_Wind_Lag_D'] = df['Jangbogo_Wind_Mean_Anomaly'].shift(daegu_lag) # 풍속 데이터 추가!
df['Month'] = df['Date'].dt.month

# 3. 정답지(Label) 생성: 평년 대비 불쾌지수가 높은 상위 20% (임계값 완화)
threshold_thi_d = df['Daegu_THI_Max_Anomaly'].quantile(0.80)
df['Is_Heatwave_Daegu'] = (df['Daegu_THI_Max_Anomaly'] >= threshold_thi_d).astype(int)

# 4. 결측치 제거 및 데이터 분리
features_d = ['Jangbogo_Lag_D', 'Jangbogo_Lag_D_MA3', 'Sejong_Lag_D', 'Jangbogo_Wind_Lag_D', 'Month']
ml_data_d = df[features_d + ['Is_Heatwave_Daegu']].dropna()

X_d = ml_data_d[features_d]
y_d = ml_data_d['Is_Heatwave_Daegu']

X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(X_d, y_d, test_size=0.2, random_state=42)

# 5. 모델 학습 (클래스 가중치 적용)
model_d = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
model_d.fit(X_train_d, y_train_d)

# 6. 예측 및 성능 평가
y_pred_d = model_d.predict(X_test_d)

print(f"=== 🚀 [대구] 폭염 예측 모델 성능 (Lag {daegu_lag}일, 풍속 추가, 상위 20%) ===")
print(f"✅ 모델 정확도(Accuracy): {accuracy_score(y_test_d, y_pred_d) * 100:.2f}%\n")
print("📊 상세 분류 리포트:")
print(classification_report(y_test_d, y_pred_d))

print("-" * 40)
print("🔍 변수 중요도")
for name, imp in zip(features_d, model_d.feature_importances_):
    print(f"🔹 {name}: {imp*100:.1f}%")

=== 🚀 [대구] 폭염 예측 모델 성능 (Lag 14일, 풍속 추가, 상위 20%) ===
✅ 모델 정확도(Accuracy): 82.86%

📊 상세 분류 리포트:
              precision    recall  f1-score   support

           0       0.83      0.98      0.90       763
           1       0.72      0.18      0.29       182

    accuracy                           0.83       945
   macro avg       0.78      0.58      0.60       945
weighted avg       0.81      0.83      0.78       945

----------------------------------------
🔍 변수 중요도
🔹 Jangbogo_Lag_D: 21.4%
🔹 Jangbogo_Lag_D_MA3: 21.5%
🔹 Sejong_Lag_D: 22.9%
🔹 Jangbogo_Wind_Lag_D: 22.5%
🔹 Month: 11.7%


## 부산

In [20]:
# 1. 부산 파생 변수 생성 (세종기지 중심, Lag 46일 적용)
busan_lag = 46

df['Sejong_Lag_B'] = df['Sejong_Temp_Mean_Anomaly'].shift(busan_lag)
df['Sejong_Lag_B_MA3'] = df['Sejong_Temp_Mean_Anomaly'].shift(busan_lag).rolling(window=3).mean()
df['Jangbogo_Lag_B'] = df['Jangbogo_Temp_Mean_Anomaly'].shift(busan_lag)
df['Sejong_Wind_Lag_B'] = df['Sejong_Wind_Mean_Anomaly'].shift(busan_lag) # 풍속 데이터 추가!

# 2. 정답지(Label) 생성: 평년 대비 불쾌지수가 높은 상위 20% (임계값 완화)
threshold_thi_b = df['Busan_THI_Max_Anomaly'].quantile(0.80)
df['Is_Heatwave_Busan'] = (df['Busan_THI_Max_Anomaly'] >= threshold_thi_b).astype(int)

# 3. 결측치 제거 및 데이터 분리
features_b = ['Sejong_Lag_B', 'Sejong_Lag_B_MA3', 'Jangbogo_Lag_B', 'Sejong_Wind_Lag_B', 'Month']
ml_data_b = df[features_b + ['Is_Heatwave_Busan']].dropna()

X_b = ml_data_b[features_b]
y_b = ml_data_b['Is_Heatwave_Busan']

X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(X_b, y_b, test_size=0.2, random_state=42)

# 4. 모델 학습 (클래스 가중치 적용)
model_b = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
model_b.fit(X_train_b, y_train_b)

# 5. 예측 및 성능 평가
y_pred_b = model_b.predict(X_test_b)

print(f"=== 🌊 [부산] 폭염 예측 모델 성능 (Lag {busan_lag}일, 풍속 추가, 상위 20%) ===")
print(f"✅ 모델 정확도(Accuracy): {accuracy_score(y_test_b, y_pred_b) * 100:.2f}%\n")
print("📊 상세 분류 리포트:")
print(classification_report(y_test_b, y_pred_b))

print("-" * 40)
print("🔍 변수 중요도")
for name, imp in zip(features_b, model_b.feature_importances_):
    print(f"🔹 {name}: {imp*100:.1f}%")

=== 🌊 [부산] 폭염 예측 모델 성능 (Lag 46일, 풍속 추가, 상위 20%) ===
✅ 모델 정확도(Accuracy): 80.72%

📊 상세 분류 리포트:
              precision    recall  f1-score   support

           0       0.81      0.98      0.89       728
           1       0.77      0.20      0.32       211

    accuracy                           0.81       939
   macro avg       0.79      0.59      0.60       939
weighted avg       0.80      0.81      0.76       939

----------------------------------------
🔍 변수 중요도
🔹 Sejong_Lag_B: 20.7%
🔹 Sejong_Lag_B_MA3: 21.1%
🔹 Jangbogo_Lag_B: 23.8%
🔹 Sejong_Wind_Lag_B: 22.5%
🔹 Month: 11.9%
